# Magnitude scale for MBWH

In this workbook, we attempt to replicate the energy-magnitude scale I implemented at MVO 2000-2003 and back-applied to 1996-1999 data

In [ ]:
# 1. Code from MVOCatalog project - probably more on hal9000 in MVO data directories

In [ ]:
import os
MVOCatalogProjectDir = os.path.join(os.getenv('HOME'), 'Developer', 'kitchensinkGT', 'PROJECTS', 'MVOcatalog', 'to_dataframes')

'''
# /Users/thompsong/Developer/kitchensinkGT/PROJECTS/MVOcatalog/to_dataframes/mbwh_subclass_magnitude_sorted2dataframe.py
import pandas as pd
df = pd.read_csv(os.path.join(MVOCatalogProjectDir, 'mbwh_subclass_magnitude_sorted.dat'),sep='\s+')
df.columns = ['year', 'month', 'day', 'hour', 'minute', 'second', 'subclass', 'emag']
df.to_csv('mbwh_subclass_magnitude_sorted.csv')
df.to_pickle(os.path.join(MVOCatalogProjectDir,'mbwh_subclass_magnitude_sorted.pkl'))

# /Users/thompsong/Developer/kitchensinkGT/PROJECTS/MVOcatalog/to_dataframes/MBWHall.aef.to_dataframe.py
import pandas as pd
df = pd.read_csv('MBWHall.aef.wronglinelengthsremoved.txt', sep='\s+')
print(df)
df.columns = ['date', 'time', 'subclass', 'amp', 'eng', 'peakf', 'F0', 'F1', 'F2', 'F3', 'F4', 'F5', 'F6', 'F7', 'F8', 'F9', 'F10', 'duration']
print(df)
df.to_csv(os.path.join(MVOCatalogProjectDir,'MBWH.aef.fixed.csv'))
df.to_pickle(os.path.join(MVOCatalogProjectDir,'MBWH.aef.fixed.pkl'))
'''


# 3. Read MBWH catalog with AEF data into Python

In [ ]:
# AEF data created by: /Users/thompsong/Dropbox/code/MVO_setups/mvocomputerbackups/Linux/_development/ampengfft
# Algorithm, for each signal:
# 1. dsignal = isignal * gain_factor
# 2. remove offset
# 3. remove instrument response (filter, FFT, deconv in f-domain, IFFT)
# 4. eng += dsignal * dsignal
# 5. abs
# 6. amp = max mean abs dsignal in 2-s moving window
# 7. write amp, eng as %8.2f
# Units of amp and eng are not clear
 
import pandas as pd
import numpy as np
import sys
sys.path.append('lib')
from importlib import reload
import dataframe_tools as DFT
reload(DFT)

In [ ]:
# AEF data generated with ampengfft. amp in um. energy in um^2? has it been divided by sample rate? (75 or 100 Hz)
csvfile = os.path.join(MVOCatalogProjectDir,'MBWH.aef.fixed.csv')
aefcat = pd.read_csv(csvfile)
A = aefcat['amp']*1e-3 # in m
R = 5000 # approx distance of MBWH from "dome" source
aefcat['ML'] = DFT.local_magnitude(A, R)
E = DFT.seismic_energy(aefcat['eng']*1e-6, R)
aefcat['ME'] = DFT.energy_magnitude(E)+1.8

DFT.summarize_catdf(aefcat)
#DFT.linregress(aefcat, 'mag', 'emag')
DFT.linearRegressMagnitudesBySubclass(aefcat, subclass_col='subclass', mag_columns=['ML', 'ME'], \
                                      subclasses=['r', 'e', 'l', 'h', 't'], plot=True, print_stats=True)

In [ ]:
print(aefcat['ML'].median())
print(aefcat['ME'].median())

In conclusion, from MBWH rows in AEF/S-files from ampengfft, we seem to get crossover at ML=ME=1.3, which is the median magnitude, if we choose:

$ML = log(amp*1e-3) + log(R) + 3.01e-6 R + 0.70$

and

$ME = 2/3 log (9.8e14 eng*1e-6 * 1/75) - 1.4$

where R is distance in m (about 5000 for MBWH), 9.8e14 is a correction for 2 pi R^2 rho speed in Boatwright, and 75 is sampling rate (since ampengfft does not divide sum(amp^2) by sample rate


# 4. Load alternative catalog that might be from merge of AEF data and MBWH mag catalog

Here we try to understand the energy magnitude scale as implemented at MVO by merging AEF data and the MBWH catalog

In [ ]:
import os
pklfile = os.path.join(MVOCatalogProjectDir, 'mbwh_subclass_aef_emag.pkl')
mergedcat = pd.read_pickle(pklfile)
'''
try:
    mergedcat = mergedcat.loc[:, ~cat.columns.str.contains('^Unnamed')]
except: 
    pass
'''
# Compute local magnitude
A = mergedcat['amp']*1e-3 # assuming ampengfft computes to displacement in mm, but could be um, or mm/s, or um/s
R = 5000 # approx distance of MBWH from "dome" source
# changing f from 1.11 to 1
#mergedcat['ML'] = np.log10(mergedcat['amp']*1000) + 1.11 * np.log10(R) + 0.00189 * R - 2.09
mergedcat['ML'] = DFT.local_magnitude(A, R)
E = DFT.seismic_energy(mergedcat['eng']*1e-6, R)
mergedcat['ME'] = DFT.energy_magnitude(E)+1.8
#mergedcat['emag2'] = np.round(0.45*np.log10(mergedcat['eng'])+2.2,1)

# remove rows with NULL values
mergedcat['emag'] = mergedcat['emag'].replace(-99.9, np.nan)
mergedcat['mag'] = mergedcat['mag'].replace(-2.0, np.nan)

mergedcat2 = mergedcat[mergedcat['emag']>=0.0]

DFT.summarize_catdf(mergedcat2)

In [ ]:
DFT.linearRegressMagnitudesBySubclass(mergedcat2, subclass_col='subclass_x', mag_columns=['emag', 'ML', 'ME'], \
                                      subclasses=['r', 'e', 'l', 'h', 't'], plot=True, print_stats=True, mfixed=[1, 1, 4/3, 4/3])

# 5. Load Machine Learning catalog

I don't think the amplitude or energy values here have been gain or response corrected, so this is a bit pointless as far as a poster tomorrow is concerned, unless i just want to plot relative sizes and rates by subclass

In [ ]:
MLcsvfile = '/home/thompsong/Developer/kitchensinkGT/PROJECTS/MVOcatalog/to_dataframes/mvo_catalog_stats.csv'
MLcat0 = pd.read_csv(MLcsvfile)

In [ ]:
MLcat = MLcat0.copy()

# Compute local magnitude
#MLcat['logamp'] = np.log10(MLcat['peakamp'])
#MLcat['logA'] = np.log10(MLcat['peakA'])
R = 5000 # approx distance of MBWH from "dome" source
MLcat['MLamp'] = DFT.local_magnitude(MLcat['peakamp'], R)
MLcat['MLA'] = DFT.local_magnitude(MLcat['peakA'], R)
#MLcat['logE'] = np.log10(MLcat['energy'])
MLcat['ME'] = DFT.energy_magnitude(MLcat['energy'], R)

'''
# remove rows with NULL values
mergedcat['emag'] = mergedcat['emag'].replace(-99.9, np.nan)
mergedcat['mag'] = mergedcat['mag'].replace(-2.0, np.nan)
'''


DFT.summarize_catdf(MLcat)

DFT.linearRegressMagnitudesBySubclass(MLcat, subclass_col='subclass', mag_columns=['MLamp', 'MLA', 'ME'], \
                                      subclasses=['r', 'e', 'l', 'h', 't'], plot=True, print_stats=True)

In [ ]:
MLcat = MLcat0.copy()

# Compute local magnitude
MLcat['logamp'] = np.log10(MLcat['peakamp'])
MLcat['logA'] = np.log10(MLcat['peakA'])
R = 5000 # approx distance of MBWH from "dome" source
MLcat['MLamp'] = np.log10(MLcat['peakamp']) + np.log10(R) + 3.01e-6 * R + 0.70
MLcat['MLA'] = np.log10(MLcat['peakA']) + np.log10(R) + 3.01e-6 * R + 0.70

MLcat['logE'] = np.log10(MLcat['energy'])
MLcat['ME'] = np.round(2/3 * MLcat['logE'],1)-1.4
'''
# remove rows with NULL values
mergedcat['emag'] = mergedcat['emag'].replace(-99.9, np.nan)
mergedcat['mag'] = mergedcat['mag'].replace(-2.0, np.nan)
'''


DFT.summarize_catdf(MLcat)

DFT.linearRegressMagnitudesBySubclass(MLcat, subclass_col='subclass', mag_columns=['MLamp', 'MLA', 'ME', 'logamp', 'logA', 'logE'], \
                                      subclasses=['r', 'e', 'l', 'h', 't'], plot=True, print_stats=True)

# Attenuation for local events and Hunga Tonga

seems that for upper limit of reasonable Q, only expect x2.1 attenuation for 10-s at MSVF and x1.1 for 100-s

for local events, not even a 0.01 magnitude change

In [ ]:
def compute_attenuation(Q, r, f, c):
    attenuation = np.exp( (-np.pi * thisf * r) / (Q * c)) 
    correction = 1/attenuation
    print('\n', f"freq={f}, R={r}, Q={Q}, speed={c}")
    print(f'traveltime={DFT.sigfigs(r/c)}')
    print(f'correction={DFT.sigfigs(correction)}')
    print(f'ML change={DFT.sigfigs(np.log10(correction))}')
    print(f'ME change={DFT.sigfigs(2/3*np.log10(correction**2))}')

Q = 100
r = 760000
f = [0.1, 0.01]
c = 3.14 * 1000
for thisf in f:
    compute_attenuation(Q, r, thisf, c)

Q = 23
r = 15000
f = 10.0
c = 1200    
compute_attenuation(Q, r, f, c)

# 6. Compute metrics for Montserrat events, including creating SDS dayfiles if needed

In [ ]:
import os
from importlib import reload
import sys
import obspy
sys.path.append('lib')
import dataframe_tools as DFT
import pipelines
reload(pipelines)
paths = {}
paths['DATA_DIR'] = os.path.join('/data')
paths['SDS_DIR'] = os.path.join(paths['DATA_DIR'], 'SDS')
paths['SAM_DIR'] = os.path.join(paths['DATA_DIR'], 'SAM')
paths['RESPONSE_DIR'] = os.path.join('data', 'responses')
print(paths)

# Montserrat data from Seisan archive
seisandbdir =  '/data/SEISAN_DB/WAV/DSNC_'
net = 'MV'
invfile = os.path.join(paths['RESPONSE_DIR'],f"{net}.xml")
source = {'lat':16.71111, 'lon':-62.17722}
dbout = 'db'
N = 3
catResultsDF = pd.DataFrame(columns=['Event', 'start', 'end', 'duration', 'ML', 'sum(ER)', 'ME', 'DR', 'DRS'])
display(catResultsDF) 

In [ ]:
def row2event(catDF, rownum):

        eventname = f'{subclass} {i}'
        startt = obspy.UTCDateTime(row['filetime'])
        endt = max([startt + min([row['trigger_duration'], 20]), row['offtime']])
        print(f'Calling for data from {startt} to {endt}')

        # need to check SDS directory exists for this date, if not, we run the sausage to create it
        pipelines.big_sausage(seisandbdir, paths, startt, endt, \
                        sampling_interval=2.56, \
                        source=source, \
                        invfile=invfile, \
                        Q=None, \
                        ext='pickle', \
                        dbout=dbout, \
                        net=net)        
        
        DFT.wrapper(paths['SDS_DIR'], net, startt, endt, invfile, catResultsDF, eventname, source)

In [ ]:
#subclasses = ['r', 'e', 'l', 'h', 't']
subclasses = ['t']
for subclass in subclasses:
    cat = MLcat[MLcat['subclass']==subclass]
    big = cat.nlargest(N, 'ME')
    for i, row in big.iterrows():
        print('\n', i, row, '\n')
        row2event(big, i)